# Notebook 05 — Real Ground-Truth Labels & Retraining (Flood + Landslide)

## Why this notebook exists

Notebooks 01 and 02 are honest about a real ceiling: `flood_occurred` and
`landslide_risk_level` are copied from **district-level** 2018 KSDMA records onto every
LSGD (or, in the ward feature store, every ward) inside that district. That means every unit
in, say, Alappuzha district has the *identical* label. A model can't learn genuine per-unit
risk from that — grouped-by-district validation correctly exposes this (flood: 50.8%,
landslide: 17.7%, both near/below what you'd get by guessing).

**More model tuning cannot fix this.** The fix has to be better labels. This notebook builds
two *real, per-unit* ground-truth sources and re-derives the targets from them:

1. **Flood** — Sentinel-1 SAR flood-extent mapping (Otsu-style threshold), computed directly
   in GEE for the Aug 2018 event. This gives a genuine **% of each unit's area that was
   actually underwater**, not a district-wide copy-paste. Method follows the published
   approach for this exact event (Uddin et al. 2019, PLOS ONE, *"Flood inundation mapping —
   Kerala 2018"*, https://doi.org/10.1371/journal.pone.0237324).
2. **Landslide** — the public, citable 2018 Kerala landslide point inventory (Hao et al. 2020,
   *Earth System Science Data*, 4,728 points, DOI: `10.17026/dans-x6c-y7x2`,
   https://doi.org/10.5194/essd-12-2899-2020). Each point is a real, field/OBIA-verified
   landslide location — spatially joining these to your LSGD/ward polygons gives a genuine
   per-unit landslide count instead of a district-wide binary flag.

Both are one-time downloads (small: a shapefile/CSV and a few GEE calls), so this slots in
before Notebooks 01/02 and doesn't touch your live rainfall pipeline at all.

**What this will *not* do:** get you to 93–97%. That number requires wall-to-wall
building-level or pixel-level ground truth for many independent events, which doesn't exist
publicly at Kerala-wide scale. What it should do is close a meaningful chunk of the gap
between your honest (~51% / ~18%) and optimistic (~73% / ~72%) numbers, because the labels
will finally vary *within* a district and not just across districts.

**Before running:** this needs `kerala_lsgd_boundaries.geojson` (Notebook 00) and
`kerala_wards_all.csv` (Notebook 00b) already in Drive, same as your other notebooks.

## 1. Setup

In [ ]:
!pip install -q geemap geopandas earthengine-api rasterio scikit-learn xgboost joblib

import ee, geemap, geopandas as gpd, pandas as pd, numpy as np
from google.colab import drive
import json, os

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/DIP_Kerala'

ee.Authenticate()
ee.Initialize(project='dip-kerala-502817')

## 2. Part A — Real flood ground truth from Sentinel-1 SAR (2018 event)

Standard approach for this exact flood (matches the published methodology cited above):
compare a pre-flood Sentinel-1 VH composite against a during-flood VH composite. Water is
much darker (lower backscatter) in VH. A fixed ~ -18 to -20 dB threshold on the during-flood
image, masked to pixels that got *darker* than the pre-flood baseline, is the standard
low-complexity version of this method — it avoids misclassifying naturally low-backscatter
land (e.g. bare soil) as flood, since it requires an actual pre/post change.

Adjust `EVENT_START` / `EVENT_END` if you want to also run this for the 2019 or 2024 events —
the function below is reusable.

In [ ]:
KERALA = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM1_NAME', 'Kerala'))
KERALA_GEOM = KERALA.geometry()

def s1_vh_composite(start, end, geom):
    coll = (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(geom)
            .filterDate(start, end)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .select('VH'))
    return coll.median().clip(geom)

def flood_extent_mask(pre_start, pre_end, flood_start, flood_end, geom, drop_db=3, water_db=-18):
    """Returns a binary flood mask: pixels that are both below `water_db` in the flood image
    AND dropped by more than `drop_db` compared to the pre-flood baseline."""
    pre = s1_vh_composite(pre_start, pre_end, geom)
    flood = s1_vh_composite(flood_start, flood_end, geom)
    dropped = pre.subtract(flood).gt(drop_db)
    is_water = flood.lt(water_db)
    return dropped.And(is_water).rename('flood')

# Pre-monsoon baseline (dry, minimal water) vs. the Aug 2018 event window
flood_mask_2018 = flood_extent_mask(
    pre_start='2018-01-01', pre_end='2018-02-28',
    flood_start='2018-08-15', flood_end='2018-08-25',
    geom=KERALA_GEOM,
)
print('Flood mask built for the Aug 2018 event.')

In [ ]:
# Sanity check: total flooded area estimate for the whole state (should be in the
# low thousands of km^2 based on published estimates for this event -- if this comes back
# as ~0 or as most of the state, the threshold/dates need adjusting before trusting the join).
area_img = flood_mask_2018.multiply(ee.Image.pixelArea())
total_flooded_km2 = area_img.reduceRegion(
    reducer=ee.Reducer.sum(), geometry=KERALA_GEOM, scale=100, maxPixels=1e10
).get('flood').getInfo()
print(f"Estimated flooded area (state-wide, Aug 2018): {total_flooded_km2/1e6:.1f} km^2")

### 2a. Zonal stats — real flood % per LSGD (polygon-based, most accurate)

In [ ]:
lsgd = gpd.read_file(f'{BASE}/00_boundaries/kerala_lsgd_boundaries.geojson')
rename_map = {'local_auth': 'lsgd_type', 'name': 'lsgd_name', 'District': 'district'}
lsgd = lsgd.rename(columns={k: v for k, v in rename_map.items() if k in lsgd.columns})
if 'lsgd_id' not in lsgd.columns:
    lsgd['lsgd_id'] = range(1, len(lsgd) + 1)

# Simplify geometry before it's serialized into the GEE request -- LSGD boundary polygons
# can carry thousands of vertices each, and reduceRegions() sends the *full* collection as
# literal geometry in one HTTP request. 1,034 unsimplified polygons blows past GEE's 10MB
# request-payload cap in a single call (this is what threw the "Request payload size
# exceeds the limit: 10485760 bytes" error). ~30m tolerance is well under the 30m pixel
# scale used below, so it doesn't meaningfully change which pixels fall inside each unit.
lsgd_simplified = lsgd[['lsgd_id', 'geometry']].copy()
lsgd_simplified['geometry'] = lsgd_simplified['geometry'].simplify(0.0003, preserve_topology=True)

# Batch into chunks so each individual request stays well under the payload limit --
# same pattern as the ward-level cell below, just smaller chunks since polygons are
# heavier than points.
CHUNK = 150
flood_pct_rows = []
for start in range(0, len(lsgd_simplified), CHUNK):
    chunk = lsgd_simplified.iloc[start:start + CHUNK]
    chunk_fc = geemap.geopandas_to_ee(chunk)
    res = flood_mask_2018.reduceRegions(
        collection=chunk_fc,
        reducer=ee.Reducer.mean(),  # mean of a 0/1 mask = fraction flooded
        scale=30,
        tileScale=4,  # reduces per-tile memory use server-side, helps avoid computation timeouts too
    )
    chunk_df = geemap.ee_to_df(res)
    flood_pct_rows.append(chunk_df)
    print(f'Processed LSGD units {start}-{start + len(chunk)}')

flood_pct_df = pd.concat(flood_pct_rows, ignore_index=True)
flood_pct_df = flood_pct_df.rename(columns={'mean': 'flood_area_pct'})[['lsgd_id', 'flood_area_pct']]
flood_pct_df['flood_area_pct'] = flood_pct_df['flood_area_pct'].fillna(0.0)
print(flood_pct_df.describe())
flood_pct_df.to_csv(f'{BASE}/03_processed/lsgd_flood_ground_truth_2018.csv', index=False)
print('Saved: lsgd_flood_ground_truth_2018.csv')

### 2b. Same thing at ward level (centroid-buffer sample, since wards are points not polygons)

Wards only have centroid coordinates (`kerala_wards_all.csv`), not surveyed polygons, so a
true zonal mean isn't possible — instead this samples the mean flood fraction inside a
~300m buffer around each ward centroid, which is a reasonable proxy at ward density.

In [ ]:
wards = pd.read_csv(f'{BASE}/00_boundaries/kerala_wards_all.csv')
wards['ward_id'] = range(1, len(wards) + 1)

ward_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([lon, lat]).buffer(300), {'ward_id': int(wid)})
    for wid, lat, lon in zip(wards['ward_id'], wards['Latitude'], wards['Longitude'])
])

# Batch this in chunks -- 21,002 buffered reduceRegions calls in one request will time out.
CHUNK = 1000
ward_flood_rows = []
for start in range(0, len(wards), CHUNK):
    chunk_fc = ee.FeatureCollection(ward_pts.toList(CHUNK, start))
    res = flood_mask_2018.reduceRegions(collection=chunk_fc, reducer=ee.Reducer.mean(), scale=30, tileScale=4)
    chunk_df = geemap.ee_to_df(res)
    ward_flood_rows.append(chunk_df)
    print(f'Processed wards {start}-{start+CHUNK}')

ward_flood_df = pd.concat(ward_flood_rows, ignore_index=True)
ward_flood_df = ward_flood_df.rename(columns={'mean': 'flood_area_pct'})[['ward_id', 'flood_area_pct']]
ward_flood_df['flood_area_pct'] = ward_flood_df['flood_area_pct'].fillna(0.0)
ward_flood_df.to_csv(f'{BASE}/03_processed/ward_flood_ground_truth_2018.csv', index=False)
print('Saved: ward_flood_ground_truth_2018.csv')
print(ward_flood_df.describe())

## 3. Part B — Real landslide ground truth from the published 2018 inventory

**Manual step (one-time):** download the point shapefile/CSV from
https://doi.org/10.17026/dans-x6c-y7x2 (Hao et al. 2020 — 4,728 verified 2018 Kerala
landslides). Upload it to `DIP_Kerala/01_labels/kerala_landslide_inventory_2018.csv` (or
`.shp` — the loader below handles either) with at minimum `latitude`/`longitude` columns.

This replaces the single district-wide `landslide_risk_level` copy-paste with a genuine
per-unit **count of verified landslides**, which is what should actually drive the label.

In [ ]:
import glob

candidates = glob.glob(f'{BASE}/01_labels/kerala_landslide_inventory_2018.*')
if not candidates:
    raise FileNotFoundError(
        "Upload the landslide inventory to DIP_Kerala/01_labels/ first -- "
        "see the markdown cell above for the download link (DOI: 10.17026/dans-x6c-y7x2)."
    )

path = candidates[0]
if path.endswith('.shp'):
    landslides = gpd.read_file(path)
    landslides['latitude'] = landslides.geometry.y
    landslides['longitude'] = landslides.geometry.x
else:
    landslides = pd.read_csv(path)
    # Normalize likely column name variants
    lat_col = next(c for c in landslides.columns if c.lower() in ('latitude', 'lat', 'y'))
    lon_col = next(c for c in landslides.columns if c.lower() in ('longitude', 'lon', 'lng', 'x'))
    landslides = landslides.rename(columns={lat_col: 'latitude', lon_col: 'longitude'})

print(f"Loaded {len(landslides)} landslide points")
landslides = gpd.GeoDataFrame(
    landslides, geometry=gpd.points_from_xy(landslides.longitude, landslides.latitude), crs='EPSG:4326'
)

In [ ]:
# 3a. LSGD-level: real point-in-polygon count per LSGD
lsgd_ls = lsgd[['lsgd_id', 'geometry']].to_crs('EPSG:4326')
joined = gpd.sjoin(landslides, lsgd_ls, how='left', predicate='within')
lsgd_landslide_counts = joined.groupby('lsgd_id').size().rename('landslide_count_real').reset_index()
lsgd_landslide_counts.to_csv(f'{BASE}/03_processed/lsgd_landslide_ground_truth_2018.csv', index=False)
print(f"LSGD units with >=1 real recorded landslide: {len(lsgd_landslide_counts)} / {len(lsgd_ls)}")

# 3b. Ward-level: count within 1km of each ward centroid (points, not polygons, so use buffer)
from scipy.spatial import cKDTree
ward_coords = np.radians(wards[['Latitude', 'Longitude']].values)
ls_coords = np.radians(landslides[['latitude', 'longitude']].values)
tree = cKDTree(ls_coords)
# ~1km in radians at Kerala's latitude
radius_rad = 1.0 / 6371.0
counts = tree.query_ball_point(ward_coords, r=radius_rad)
wards['landslide_count_real'] = [len(c) for c in counts]
wards[['ward_id', 'landslide_count_real']].to_csv(
    f'{BASE}/03_processed/ward_landslide_ground_truth_2018.csv', index=False
)
print(wards['landslide_count_real'].describe())

## 4. Extra discriminative features (address the documented gap in Notebook 02)

Notebook 02 explicitly flagged soil type as a missing predictor. Adding it here, plus a
drainage/flow-accumulation proxy (HydroSHEDS), which is a stronger stability signal than
`dist_to_water_m` alone.

In [ ]:
soil = ee.Image('OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02').select('b0').rename('soil_texture_class')
flow_acc = ee.Image('WWF/HydroSHEDS/15ACC').select('b1').rename('flow_accumulation')

# Two separate reductions rather than one combined reducer -- combine(sharedInputs=True)
# applies BOTH reducers to BOTH bands, so it actually outputs soil_texture_class_mode,
# soil_texture_class_mean, flow_accumulation_mode, flow_accumulation_mean rather than the
# plain column names used below. Soil texture is categorical (mode makes sense), flow
# accumulation is continuous (mean makes sense) -- keeping them separate also avoids
# computing a meaningless mean-of-a-class-id or mode-of-a-continuous-value for nothing.
soil_rows, flow_rows = [], []
for start in range(0, len(lsgd_simplified), CHUNK):
    chunk = lsgd_simplified.iloc[start:start + CHUNK]
    chunk_fc = geemap.geopandas_to_ee(chunk)

    soil_res = soil.reduceRegions(collection=chunk_fc, reducer=ee.Reducer.mode(), scale=90, tileScale=4)
    soil_rows.append(geemap.ee_to_df(soil_res))

    flow_res = flow_acc.reduceRegions(collection=chunk_fc, reducer=ee.Reducer.mean(), scale=90, tileScale=4)
    flow_rows.append(geemap.ee_to_df(flow_res))

    print(f'Processed LSGD units {start}-{start + len(chunk)} (soil/drainage)')

soil_df = pd.concat(soil_rows, ignore_index=True).rename(columns={'mode': 'soil_texture_class'})[['lsgd_id', 'soil_texture_class']]
flow_df = pd.concat(flow_rows, ignore_index=True).rename(columns={'mean': 'flow_accumulation'})[['lsgd_id', 'flow_accumulation']]
lsgd_extra = soil_df.merge(flow_df, on='lsgd_id', how='outer')

lsgd_extra.to_csv(f'{BASE}/03_processed/lsgd_extra_features.csv', index=False)
print('Saved lsgd_extra_features.csv (soil_texture_class, flow_accumulation)')
lsgd_extra.head()

## 5. Rebuild the feature stores with real labels

Merges the existing terrain/rainfall features with the new real flood %, real landslide
count, and the extra soil/drainage features. Produces `lsgd_feature_store_v2.csv` and
`ward_feature_store_v2.csv` — the *_v2 suffix is deliberate so your existing v1 files and
Notebook 04's live-query logic aren't touched until you've validated these.

In [ ]:
# --- LSGD v2 ---
fs = pd.read_csv(f'{BASE}/03_processed/lsgd_feature_store.csv')
flood_gt = pd.read_csv(f'{BASE}/03_processed/lsgd_flood_ground_truth_2018.csv')
ls_gt = pd.read_csv(f'{BASE}/03_processed/lsgd_landslide_ground_truth_2018.csv')
extra = pd.read_csv(f'{BASE}/03_processed/lsgd_extra_features.csv')

fs_v2 = fs.merge(flood_gt, on='lsgd_id', how='left') \
          .merge(ls_gt, on='lsgd_id', how='left') \
          .merge(extra[['lsgd_id', 'soil_texture_class', 'flow_accumulation']], on='lsgd_id', how='left')

fs_v2['landslide_count_real'] = fs_v2['landslide_count_real'].fillna(0)
fs_v2['flood_area_pct'] = fs_v2['flood_area_pct'].fillna(0.0)

# Real, per-unit targets -- replacing the district-copy-paste versions
fs_v2['flood_occurred_real'] = (fs_v2['flood_area_pct'] > 0.05).astype(int)  # >5% of area underwater
fs_v2['landslide_occurred_real'] = (fs_v2['landslide_count_real'] > 0).astype(int)

fs_v2.to_csv(f'{BASE}/03_processed/lsgd_feature_store_v2.csv', index=False)
print('lsgd_feature_store_v2.csv:', fs_v2.shape)
print(fs_v2[['flood_occurred', 'flood_occurred_real']].apply(pd.Series.value_counts))
print(fs_v2[['landslide_occurred', 'landslide_occurred_real']].apply(pd.Series.value_counts))

In [ ]:
# --- Ward v2 (same pattern) ---
ward_fs = pd.read_csv(f'{BASE}/03_processed/ward_feature_store.csv')
ward_flood_gt = pd.read_csv(f'{BASE}/03_processed/ward_flood_ground_truth_2018.csv')
ward_ls_gt = pd.read_csv(f'{BASE}/03_processed/ward_landslide_ground_truth_2018.csv')

ward_fs_v2 = ward_fs.merge(ward_flood_gt, on='ward_id', how='left') \
                     .merge(ward_ls_gt, on='ward_id', how='left')
ward_fs_v2['flood_area_pct'] = ward_fs_v2['flood_area_pct'].fillna(0.0)
ward_fs_v2['landslide_count_real'] = ward_fs_v2['landslide_count_real'].fillna(0)
ward_fs_v2['flood_occurred_real'] = (ward_fs_v2['flood_area_pct'] > 0.05).astype(int)
ward_fs_v2['landslide_occurred_real'] = (ward_fs_v2['landslide_count_real'] > 0).astype(int)

ward_fs_v2.to_csv(f'{BASE}/03_processed/ward_feature_store_v2.csv', index=False)
print('ward_feature_store_v2.csv:', ward_fs_v2.shape)

## 6. Retrain flood model on real labels

Same model family / hyperparameters as Notebook 01 (RandomForest / XGBoost / LightGBM) so
the comparison to the old honest number is apples-to-apples -- only the labels and two extra
features changed. Still reports BOTH a random split and a district-grouped split. If real
per-unit labels are doing their job, the grouped-split number should stop being anomalously
close to (or below) chance, because units *within* the same district now legitimately
disagree with each other.

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.base import clone
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings

feature_cols_v2 = ['elevation', 'slope', 'rainfall_7day_mm', 'dist_to_water_m',
                    'vegetation', 'builtup', 'flow_accumulation']
target_col = 'flood_occurred_real'

df2 = fs_v2.dropna(subset=feature_cols_v2 + [target_col])
X2 = df2[feature_cols_v2]
y2 = df2[target_col].astype(int)
print(X2.shape, y2.value_counts().to_dict())

X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)
scaler2 = StandardScaler()
X_train_s, X_test_s = scaler2.fit_transform(X_train), scaler2.transform(X_test)

models2 = {
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42, verbose=-1),
}
results2 = {}
for name, model in models2.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    probs = model.predict_proba(X_test_s)[:, 1]
    auc = roc_auc_score(y_test, probs) if len(set(y_test)) > 1 else float('nan')
    results2[name] = {'model': model, 'preds': preds, 'auc': auc}
    print(f"\n===== {name} =====")
    print(classification_report(y_test, preds, zero_division=0))
    print('ROC-AUC:', round(auc, 4) if auc == auc else 'n/a')

valid2 = {k: v for k, v in results2.items() if v['auc'] == v['auc']}
best_name2 = max(valid2, key=lambda k: valid2[k]['auc']) if valid2 else list(results2.keys())[0]
best_model2 = results2[best_name2]['model']

groups2 = df2.loc[X2.index, 'district']
accs2, f1s2 = [], []
from sklearn.metrics import f1_score
for seed in range(30):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr, te = next(gss.split(X2, y2, groups=groups2))
    if len(set(y2.iloc[te])) < 2:
        continue
    sc = StandardScaler()
    Xtr_s, Xte_s = sc.fit_transform(X2.iloc[tr]), sc.transform(X2.iloc[te])
    m = clone(models2[best_name2])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        m.fit(Xtr_s, y2.iloc[tr])
    preds_te = m.predict(Xte_s)
    accs2.append((preds_te == y2.iloc[te].values).mean())
    # Macro-F1 alongside accuracy: the real labels turned out far more imbalanced
    # (~79/21) than the old district-copy ones (~58/42), so a majority-class baseline
    # alone now scores ~79%% -- accuracy by itself overstates how good the model is.
    # Macro-F1 weights both classes equally and won't reward that shortcut.
    f1s2.append(f1_score(y2.iloc[te].values, preds_te, average='macro', zero_division=0))

print(f"\nBest model: {best_name2}")
print(f"NEW honest district-grouped accuracy: {np.mean(accs2):.4f} (was 0.5084 with district-copy labels)")
print(f"NEW honest district-grouped macro-F1: {np.mean(f1s2):.4f} (std: {np.std(f1s2):.4f})")
majority_baseline = max(y2.mean(), 1 - y2.mean())
print(f"For reference, always guessing the majority class gets {majority_baseline:.4f} accuracy --")
print(f"compare the model's macro-F1 above to this, not just its accuracy.")

os.makedirs(f'{BASE}/05_models', exist_ok=True)
joblib.dump(best_model2, f'{BASE}/05_models/flood_model_v2.pkl')
joblib.dump(scaler2, f'{BASE}/05_models/flood_scaler_v2.pkl')
json.dump({
    'model': best_name2,
    'features': feature_cols_v2,
    'random_split_accuracy': float((results2[best_name2]['preds'] == y_test.values).mean()),
    'honest_grouped_accuracy_mean': float(np.mean(accs2)),
    'honest_grouped_accuracy_std': float(np.std(accs2)),
    'honest_grouped_macro_f1_mean': float(np.mean(f1s2)),
    'honest_grouped_macro_f1_std': float(np.std(f1s2)),
    'majority_class_baseline_accuracy': float(majority_baseline),
    'label_source': 'Sentinel-1 SAR flood extent (real per-unit), not district copy-paste',
}, open(f'{BASE}/05_models/flood_accuracy_v2.json', 'w'), indent=2)
print('Saved flood_model_v2.pkl + flood_accuracy_v2.json')

## 7. Retrain landslide model on real labels

Same pattern -- swap `landslide_risk_level` (district copy-paste) for a model trained on
`landslide_count_real` binned into risk tiers, so you keep a multi-class output for the UI
but the ground truth is now real per-unit landslide presence/density.

In [ ]:
feature_cols_ls2 = ['elevation', 'slope', 'rainfall_7day_mm', 'vegetation', 'dist_to_water_m', 'soil_texture_class']

def bin_landslide_risk(count):
    if count == 0: return 'Low'
    if count <= 2: return 'Moderate'
    if count <= 6: return 'High'
    return 'Critical'

fs_v2['landslide_risk_level_real'] = fs_v2['landslide_count_real'].apply(bin_landslide_risk)

from sklearn.preprocessing import LabelEncoder
df_ls2 = fs_v2.dropna(subset=feature_cols_ls2 + ['landslide_risk_level_real'])
X_ls2 = df_ls2[feature_cols_ls2]
le2 = LabelEncoder()
y_ls2 = le2.fit_transform(df_ls2['landslide_risk_level_real'])
print('Class distribution:', pd.Series(df_ls2['landslide_risk_level_real']).value_counts().to_dict())

Xtr, Xte, ytr, yte = train_test_split(X_ls2, y_ls2, test_size=0.2, random_state=42, stratify=y_ls2)
sc_ls2 = StandardScaler()
Xtr_s, Xte_s = sc_ls2.fit_transform(Xtr), sc_ls2.transform(Xte)

ls_model2 = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42, eval_metric='mlogloss')
ls_model2.fit(Xtr_s, ytr)
preds = ls_model2.predict(Xte_s)
print(classification_report(yte, preds, target_names=le2.classes_, zero_division=0))

groups_ls2 = df_ls2.loc[X_ls2.index, 'district']
accs_ls2, f1s_ls2 = [], []
for seed in range(30):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr, te = next(gss.split(X_ls2, y_ls2, groups=groups_ls2))
    if len(set(y_ls2[te])) < 2:
        continue
    sc = StandardScaler()
    Xtr_s2, Xte_s2 = sc.fit_transform(X_ls2.iloc[tr]), sc.transform(X_ls2.iloc[te])
    m = clone(ls_model2)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        m.fit(Xtr_s2, y_ls2[tr])
    preds_te2 = m.predict(Xte_s2)
    accs_ls2.append((preds_te2 == y_ls2[te]).mean())
    # Same reasoning as the flood cell above -- the real landslide labels are heavily
    # skewed toward 'Low' (862 of 1,034 LSGDs), so accuracy alone rewards a model that
    # just predicts 'Low' every time. Macro-F1 catches that; watch it especially for
    # the High/Moderate tiers, which the single-split run above showed near 0 recall on.
    f1s_ls2.append(f1_score(y_ls2[te], preds_te2, average='macro', zero_division=0))

print(f"\nNEW honest district-grouped accuracy: {np.mean(accs_ls2):.4f} (was 0.1773 with district-copy labels)")
print(f"NEW honest district-grouped macro-F1: {np.mean(f1s_ls2):.4f} (std: {np.std(f1s_ls2):.4f})")
ls_majority_baseline = df_ls2['landslide_risk_level_real'].value_counts(normalize=True).max()
print(f"For reference, always guessing the majority class ('Low') gets {ls_majority_baseline:.4f} accuracy --")
print(f"compare the model's macro-F1 above to this, not just its accuracy.")

joblib.dump(ls_model2, f'{BASE}/05_models/landslide_model_v2.pkl')
joblib.dump(sc_ls2, f'{BASE}/05_models/landslide_scaler_v2.pkl')
joblib.dump(le2, f'{BASE}/05_models/landslide_label_encoder_v2.pkl')
json.dump({
    'model': 'XGBoost',
    'features': feature_cols_ls2,
    'honest_grouped_accuracy_mean': float(np.mean(accs_ls2)),
    'honest_grouped_accuracy_std': float(np.std(accs_ls2)),
    'honest_grouped_macro_f1_mean': float(np.mean(f1s_ls2)),
    'honest_grouped_macro_f1_std': float(np.std(f1s_ls2)),
    'majority_class_baseline_accuracy': float(ls_majority_baseline),
    'label_source': f'Hao et al. 2020 verified inventory ({len(landslides)} points), spatial join, not district copy-paste',
}, open(f'{BASE}/05_models/landslide_accuracy_v2.json', 'w'), indent=2)
print('Saved landslide_model_v2.pkl + landslide_accuracy_v2.json')

## 8. Before/after summary — report this table honestly, same as before

Once both cells above have run, this prints a clean comparison for your report.

In [ ]:
print(f"{'Model':<12}{'Old honest acc':<18}{'New honest acc':<18}{'New macro-F1':<15}{'Majority baseline'}")
print(f"{'Flood':<12}{'0.5084':<18}{np.mean(accs2):<18.4f}{np.mean(f1s2):<15.4f}{majority_baseline:.4f}")
print(f"{'Landslide':<12}{'0.1773':<18}{np.mean(accs_ls2):<18.4f}{np.mean(f1s_ls2):<15.4f}{ls_majority_baseline:.4f}")
print("\nReport macro-F1 as the headline honest number, not accuracy -- with the real labels")
print("turning out this imbalanced, accuracy alone can look good just from guessing the")
print("majority class (compare it to the baseline column). This is the same discipline")
print("Notebook 03 already applies to the damage assessment model.")
print("\nStill state plainly in your report: this is one event-year (2018) for flood ground")
print("truth and one inventory (2018) for landslide -- more years of real inundation/inventory")
print("data would further reduce variance, same 'more data > more tuning' principle as before.")

## 9. Wiring this into Notebook 04 / the backend

Once you've validated `flood_model_v2.pkl` / `landslide_model_v2.pkl` beat the old honest
numbers, swap them into `backend/services/inference.py` and `backend/ml_models/` — same
filenames pattern the app already expects, just with `_v2` (or rename to replace the
originals once you're confident). Don't touch `04_Combined_Live_Risk_Query.ipynb`'s
live-rainfall logic — only the static feature lookup + which `.pkl` gets loaded changes.